# Experiment 3 - Balanced Random Forest vs Random Forest

## Alternative 2: Balanced Random Forest vs Random Forest

As the 2nd Alternative approach a comparison of different class balancing methods using
random Forst. 3 models were compared Random Forest without SMOTE, Random Forest with
SMOTE and Balance Random Forest. Balance random forest is a modified version of random
forest that is good in handling imbalance classes. It handles the class imbalance by selecting
equal numbers of samples from each class during training its decision trees. This
is why Balanced Random Forest was proposed as an alternative approach because the BRFSS
study dataset also had a large imbalance in the target diabetes column. It helps to investigate how
class imbalance affects diabetes prediction and check whether Balanced Random Forest could
improve the validation results.


In [1]:
import pandas as pd
import numpy as np
import joblib
from pathlib import Path


In [2]:
dt = joblib.load("starting_split.joblib")
x_train = dt["x_train"]
y_train = dt["y_train"]


In [3]:
x_train.shape


(4507, 23)

In [4]:
y_train.value_counts()


diabetes
0.0    3685
1.0     822
Name: count, dtype: int64

In [5]:
OUTPUT = Path("results/experiment3_balanced_random_forest")
OUTPUT.mkdir(parents=True, exist_ok=True)
MODEL_DIR = OUTPUT / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)


### MICE imputation

Saved MICE imputed on training and validation data for each fold.


In [6]:
def apply_mice(train_data, validation_data, folder_name):
    folder = Path(folder_name)
    frames = []
    for filename, original in [('train_imputed.csv', train_data),('validation_imputed.csv', validation_data)]:
        path = folder / filename
        if not path.exists():
            raise FileNotFoundError(f'Missing {path}')
        frame = pd.read_csv(path)
        if frame.isna().any().any():
            raise ValueError(f'Missing values in baseline MICE data: {path}')
        frame.index = original.index
        frames.append(frame)
    return tuple(frames)


### Prepare five folds with OneHotEncoder and RFE20


In [7]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.feature_selection import RFE
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

In [ ]:
prep_folds = []
sel_feats = []

for fold_num, (train_idx, val_idx) in enumerate(cv.split(x_train, y_train), start=1):
    print("Preparing fold", fold_num, "/5")
    x_tr = x_train.iloc[train_idx]
    x_val = x_train.iloc[val_idx]
    y_tr = y_train.iloc[train_idx]
    y_val = y_train.iloc[val_idx]

    x_tr_imp, x_val_imp = apply_mice( x_tr, x_val, f"mice_cache/fold_{fold_num}")

    encoder = OneHotEncoder(handle_unknown="ignore", drop=None, sparse_output=False)
    x_tr_enc = encoder.fit_transform(x_tr_imp.astype(str))
    x_val_enc = encoder.transform(x_val_imp.astype(str))
    print("Training shape after OneHotEncoder:", x_tr_enc.shape)
    print("Validation shape after OneHotEncoder:", x_val_enc.shape)

    sel_lr = LogisticRegression(solver="liblinear", max_iter=1000, random_state=42)
    selector = RFE(estimator=sel_lr, n_features_to_select=20, step=1)
    x_tr_sel = selector.fit_transform(x_tr_enc, y_tr)
    x_val_sel = selector.transform(x_val_enc)
    print("Training shape after RFE20:", x_tr_sel.shape)

    feature_names = encoder.get_feature_names_out()[selector.support_]
    for feature_name in feature_names:
        sel_feats.append({"fold": fold_num, "feature": feature_name})

    prep_folds.append({"fold": fold_num, "encoder": encoder, "selector": selector,"x_train": x_tr_sel, "y_train": y_tr.to_numpy(),
        "x_val": x_val_sel, "y_val": y_val.to_numpy(),"features": x_tr_sel.shape[1]})


Preparing fold 1 /5
Training shape after OneHotEncoder: (3605, 68)
Validation shape after OneHotEncoder: (902, 68)
Training shape after RFE20: (3605, 20)
Preparing fold 2 /5
Training shape after OneHotEncoder: (3605, 68)
Validation shape after OneHotEncoder: (902, 68)
Training shape after RFE20: (3605, 20)
Preparing fold 3 /5
Training shape after OneHotEncoder: (3606, 68)
Validation shape after OneHotEncoder: (901, 68)
Training shape after RFE20: (3606, 20)
Preparing fold 4 /5
Training shape after OneHotEncoder: (3606, 68)
Validation shape after OneHotEncoder: (901, 68)
Training shape after RFE20: (3606, 20)
Preparing fold 5 /5
Training shape after OneHotEncoder: (3606, 68)
Validation shape after OneHotEncoder: (901, 68)
Training shape after RFE20: (3606, 20)


In [9]:
pd.DataFrame(sel_feats).to_csv(OUTPUT / "rfe20_selected_features.csv", index=False)

In [10]:
print("RFE20 ready:", len(prep_folds), "folds")

RFE20 ready: 5 folds


### Apply SMOTE to training data only

Create balanced training folds for Random Forest with SMOTE. Random Forest without balancing and Balanced Random Forest use the original training folds. All three use the same validation data.


In [11]:
# Apply SMOTE to the training data only.
import os
os.environ.setdefault("LOKY_MAX_CPU_COUNT", str(os.cpu_count() or 1))
from imblearn.over_sampling import SMOTE

smote_folds = []

for fold in prep_folds:
    smote = SMOTE(random_state=42)
    x_bal, y_bal = smote.fit_resample(fold["x_train"], fold["y_train"])
    smote_folds.append({
        "fold": fold["fold"], "encoder": fold["encoder"], "selector": fold["selector"],
        "x_train": x_bal, "y_train": y_bal,"x_val": fold["x_val"], "y_val": fold["y_val"], "features": fold["features"]})


c:\Users\chali\AppData\Local\Programs\Python\Python311\Lib\site-packages\joblib\externals\loky\backend\context.py:136: UserWarning: Could not find the number of physical cores for the following reason:
[WinError 2] The system cannot find the file specified
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "c:\Users\chali\AppData\Local\Programs\Python\Python311\Lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
               ^^^^^^^^^^^^^^^
  File "c:\Users\chali\AppData\Local\Programs\Python\Python311\Lib\subprocess.py", line 546, in run
    with Popen(*popenargs, **kwargs) as process:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\chali\AppData\Local\Programs\Python\Python311\Lib\subprocess.py", line 1022, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
  

# Model training

For all 3 Rf model same best hyper parameters which was identified during
my baseline was used to keep the model comparison fair between all three.

All three models use the same settings: 200 trees, `max_depth=5`, `min_samples_split=2`, `max_features="log2"` and `random_state=42`. Balanced Random Forest also uses `sampling_strategy="all"`, `replacement=True` and `bootstrap=False` to balance the classes during training.


## 1. Random Forest without SMOTE


In [12]:
from sklearn.ensemble import RandomForestClassifier


### 1.1 Train the model


In [13]:
rf_models = []
rf_preds = []
rf_probs = []

for fold_num, fold in enumerate(prep_folds, start=1):
    print("A. Random Forest: training fold", fold_num, "/5")
    model = RandomForestClassifier(n_estimators=200, max_depth=5, min_samples_split=2, max_features="log2", random_state=42)
    model.fit(fold["x_train"], fold["y_train"])

    y_pred = model.predict(fold["x_val"]).ravel()
    y_prob = model.predict_proba(fold["x_val"])[:, 1]
    rf_preds.append(y_pred)
    rf_probs.append(y_prob)

    rf_models.append({"fold": fold_num, "model": model, "encoder": fold["encoder"],
        "selector": fold["selector"], "input_columns": list(x_train.columns)})


A. Random Forest: training fold 1 /5
A. Random Forest: training fold 2 /5
A. Random Forest: training fold 3 /5
A. Random Forest: training fold 4 /5
A. Random Forest: training fold 5 /5


### 1.2 Calculate validation metrics


In [14]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, precision_recall_curve, auc, balanced_accuracy_score
)
from IPython.display import display

met_cols = ["accuracy", "precision", "recall", "f1", "roc_auc", "pr_auc", "balanced_accuracy", "specificity"]


In [15]:
rf_met = pd.DataFrame(columns=["fold", "features"] + met_cols)

for i, fold in enumerate(prep_folds):
    y_true = fold["y_val"]
    y_pred = rf_preds[i]
    y_prob = rf_probs[i]
    precision, recall, _ = precision_recall_curve(y_true, y_prob, pos_label=1)

    rf_fold_acc = accuracy_score(y_true, y_pred)
    rf_fold_prec = precision_score(y_true, y_pred, zero_division=0)
    rf_fold_rec = recall_score(y_true, y_pred)
    rf_fold_f1 = f1_score(y_true, y_pred)
    rf_fold_roc_auc = roc_auc_score(y_true, y_prob)
    rf_fold_pr_auc = auc(recall, precision)
    rf_fold_bal_acc = balanced_accuracy_score(y_true, y_pred)
    rf_fold_spec = recall_score(y_true, y_pred, pos_label=0)

    rf_met.loc[i] = [i + 1, fold["features"], rf_fold_acc, rf_fold_prec,
        rf_fold_rec, rf_fold_f1, rf_fold_roc_auc,
        rf_fold_pr_auc, rf_fold_bal_acc, rf_fold_spec ]


### 1.3 Calculate the mean and standard deviation across five folds


In [16]:
rf_acc = rf_met["accuracy"].mean()
rf_prec = rf_met["precision"].mean()
rf_rec = rf_met["recall"].mean()
rf_f1 = rf_met["f1"].mean()
rf_roc_auc = rf_met["roc_auc"].mean()
rf_pr_auc = rf_met["pr_auc"].mean()
rf_bal_acc = rf_met["balanced_accuracy"].mean()
rf_spec = rf_met["specificity"].mean()

rf_sum = rf_met[met_cols].agg(["mean", "std"]).T


In [17]:
display(rf_met)


,fold,features,accuracy,precision,recall,f1,roc_auc,pr_auc,balanced_accuracy,specificity
0,1.0,20.0,0.817073,0.500000,0.036364,0.067797,0.790000,0.410553,0.514111,0.991859
1,2.0,20.0,0.825942,0.653846,0.103030,0.178010,0.757025,0.436706,0.545409,0.987788
2,3.0,20.0,0.825749,0.888889,0.048780,0.092486,0.809201,0.528105,0.523712,0.998643
3,4.0,20.0,0.827969,0.846154,0.067073,0.124294,0.774700,0.463084,0.532180,0.997286
4,5.0,20.0,0.820200,0.600000,0.036585,0.068966,0.787959,0.463490,0.515579,0.994573


In [18]:
display(rf_sum)


,mean,std
accuracy,0.823387,0.004559
precision,0.697778,0.165186
recall,0.058367,0.027933
f1,0.106310,0.046188
roc_auc,0.783777,0.019369
pr_auc,0.460387,0.043723
balanced_accuracy,0.526198,0.012941
specificity,0.994030,0.004355


### 1.4 Save the fold models and results


In [19]:
rf_met.to_csv(OUTPUT / "rf_fold_metrics.csv", index=False)
rf_sum.to_csv(OUTPUT / "rf_summary.csv")
joblib.dump({"variant": "rf", "positive_class": 1,"requires_mice_imputed_input": True, "fold_models": rf_models
}, MODEL_DIR / "rf.joblib")
print("Saved A. Random Forest fold models and results")


Saved A. Random Forest fold models and results


## 2. Random Forest with SMOTE


In [20]:
from sklearn.ensemble import RandomForestClassifier


### 2.1 Train the model


In [21]:
rf_smote_models = []
rf_smote_preds = []
rf_smote_probs = []

for fold_num, fold in enumerate(smote_folds, start=1):
    print("B. Random Forest + SMOTE: training fold", fold_num, "/5")
    model = RandomForestClassifier(n_estimators=200, max_depth=5, min_samples_split=2, max_features="log2", random_state=42)
    model.fit(fold["x_train"], fold["y_train"])

    y_pred = model.predict(fold["x_val"]).ravel()
    y_prob = model.predict_proba(fold["x_val"])[:, 1]
    rf_smote_preds.append(y_pred)
    rf_smote_probs.append(y_prob)

    rf_smote_models.append({"fold": fold_num, "model": model, "encoder": fold["encoder"],"selector": fold["selector"], "input_columns": list(x_train.columns) })


B. Random Forest + SMOTE: training fold 1 /5
B. Random Forest + SMOTE: training fold 2 /5
B. Random Forest + SMOTE: training fold 3 /5
B. Random Forest + SMOTE: training fold 4 /5
B. Random Forest + SMOTE: training fold 5 /5


### 2.2 Calculate validation metrics


In [22]:
rf_smote_met = pd.DataFrame(columns=["fold", "features"] + met_cols)

for i, fold in enumerate(smote_folds):
    y_true = fold["y_val"]
    y_pred = rf_smote_preds[i]
    y_prob = rf_smote_probs[i]
    precision, recall, _ = precision_recall_curve(y_true, y_prob, pos_label=1)

    rf_smote_fold_acc = accuracy_score(y_true, y_pred)
    rf_smote_fold_prec = precision_score(y_true, y_pred, zero_division=0)
    rf_smote_fold_rec = recall_score(y_true, y_pred)
    rf_smote_fold_f1 = f1_score(y_true, y_pred)
    rf_smote_fold_roc_auc = roc_auc_score(y_true, y_prob)
    rf_smote_fold_pr_auc = auc(recall, precision)
    rf_smote_fold_bal_acc = balanced_accuracy_score(y_true, y_pred)
    rf_smote_fold_spec = recall_score(y_true, y_pred, pos_label=0)

    rf_smote_met.loc[i] = [ i + 1, fold["features"], rf_smote_fold_acc, rf_smote_fold_prec,
        rf_smote_fold_rec, rf_smote_fold_f1, rf_smote_fold_roc_auc,
        rf_smote_fold_pr_auc, rf_smote_fold_bal_acc, rf_smote_fold_spec ]


### 2.3 Calculate the mean and standard deviation across five folds


In [23]:
rf_smote_acc = rf_smote_met["accuracy"].mean()
rf_smote_prec = rf_smote_met["precision"].mean()
rf_smote_rec = rf_smote_met["recall"].mean()
rf_smote_f1 = rf_smote_met["f1"].mean()
rf_smote_roc_auc = rf_smote_met["roc_auc"].mean()
rf_smote_pr_auc = rf_smote_met["pr_auc"].mean()
rf_smote_bal_acc = rf_smote_met["balanced_accuracy"].mean()
rf_smote_spec = rf_smote_met["specificity"].mean()

rf_smote_sum = rf_smote_met[met_cols].agg(["mean", "std"]).T


In [24]:
display(rf_smote_met)


,fold,features,accuracy,precision,recall,f1,roc_auc,pr_auc,balanced_accuracy,specificity
0,1.0,20.0,0.733925,0.377049,0.696970,0.489362,0.783134,0.405143,0.719584,0.742198
1,2.0,20.0,0.710643,0.350932,0.684848,0.464066,0.757781,0.424366,0.700633,0.716418
2,3.0,20.0,0.730300,0.374603,0.719512,0.492693,0.799753,0.506326,0.726106,0.732700
3,4.0,20.0,0.702553,0.341463,0.682927,0.455285,0.763358,0.441756,0.694923,0.706920
4,5.0,20.0,0.735849,0.380645,0.719512,0.497890,0.790126,0.461129,0.729498,0.739484


In [25]:
display(rf_smote_sum)


,mean,std
accuracy,0.722654,0.015066
precision,0.364939,0.017564
recall,0.700754,0.017950
f1,0.479859,0.018931
roc_auc,0.778830,0.017793
pr_auc,0.447744,0.038757
balanced_accuracy,0.714149,0.015495
specificity,0.727544,0.015273


### 2.4 Save the fold models and results


In [26]:
rf_smote_met.to_csv(OUTPUT / "rf_smote_fold_metrics.csv", index=False)
rf_smote_sum.to_csv(OUTPUT / "rf_smote_summary.csv")
joblib.dump({"variant": "rf_smote", "positive_class": 1,"requires_mice_imputed_input": True, "fold_models": rf_smote_models}, MODEL_DIR / "rf_smote.joblib")
print("Saved B. Random Forest + SMOTE fold models and results")


Saved B. Random Forest + SMOTE fold models and results


## 3. Balanced Random Forest


In [27]:
from imblearn.ensemble import BalancedRandomForestClassifier


### 3.1 Train the model


In [28]:
# Train Balanced Random Forest without external SMOTE.
brf_models = []
brf_preds = []
brf_probs = []

for fold_num, fold in enumerate(prep_folds, start=1):
    print("C. Balanced Random Forest: training fold", fold_num, "/5")
    model = BalancedRandomForestClassifier(n_estimators=200, max_depth=5, min_samples_split=2, max_features="log2", random_state=42, sampling_strategy="all", replacement=True, bootstrap=False)
    model.fit(fold["x_train"], fold["y_train"])

    y_pred = model.predict(fold["x_val"]).ravel()
    y_prob = model.predict_proba(fold["x_val"])[:, 1]
    brf_preds.append(y_pred)
    brf_probs.append(y_prob)

    brf_models.append({"fold": fold_num, "model": model, "encoder": fold["encoder"], "selector": fold["selector"], "input_columns": list(x_train.columns)})


C. Balanced Random Forest: training fold 1 /5
C. Balanced Random Forest: training fold 2 /5
C. Balanced Random Forest: training fold 3 /5
C. Balanced Random Forest: training fold 4 /5
C. Balanced Random Forest: training fold 5 /5


### 3.2 Calculate validation metrics


In [29]:
brf_met = pd.DataFrame(columns=["fold", "features"] + met_cols)

for i, fold in enumerate(prep_folds):
    y_true = fold["y_val"]
    y_pred = brf_preds[i]
    y_prob = brf_probs[i]
    precision, recall, _ = precision_recall_curve(y_true, y_prob, pos_label=1)

    brf_fold_acc = accuracy_score(y_true, y_pred)
    brf_fold_prec = precision_score(y_true, y_pred, zero_division=0)
    brf_fold_rec = recall_score(y_true, y_pred)
    brf_fold_f1 = f1_score(y_true, y_pred)
    brf_fold_roc_auc = roc_auc_score(y_true, y_prob)
    brf_fold_pr_auc = auc(recall, precision)
    brf_fold_bal_acc = balanced_accuracy_score(y_true, y_pred)
    brf_fold_spec = recall_score(y_true, y_pred, pos_label=0)

    brf_met.loc[i] = [ i + 1, fold["features"], brf_fold_acc, brf_fold_prec,
        brf_fold_rec, brf_fold_f1, brf_fold_roc_auc,
        brf_fold_pr_auc, brf_fold_bal_acc, brf_fold_spec]


### 3.3 Calculate the mean and standard deviation across five folds


In [30]:
brf_acc = brf_met["accuracy"].mean()
brf_prec = brf_met["precision"].mean()
brf_rec = brf_met["recall"].mean()
brf_f1 = brf_met["f1"].mean()
brf_roc_auc = brf_met["roc_auc"].mean()
brf_pr_auc = brf_met["pr_auc"].mean()
brf_bal_acc = brf_met["balanced_accuracy"].mean()
brf_spec = brf_met["specificity"].mean()

brf_sum = brf_met[met_cols].agg(["mean", "std"]).T


In [31]:
display(brf_met)


,fold,features,accuracy,precision,recall,f1,roc_auc,pr_auc,balanced_accuracy,specificity
0,1.0,20.0,0.720621,0.366154,0.721212,0.485714,0.789334,0.408859,0.720850,0.720488
1,2.0,20.0,0.678492,0.331536,0.745455,0.458955,0.757970,0.427169,0.704478,0.663501
2,3.0,20.0,0.701443,0.350427,0.750000,0.477670,0.810674,0.529103,0.720319,0.690638
3,4.0,20.0,0.688124,0.335211,0.725610,0.458574,0.773683,0.459213,0.702696,0.679783
4,5.0,20.0,0.685905,0.339623,0.768293,0.471028,0.791152,0.465486,0.717932,0.667571


In [32]:
display(brf_sum)


,mean,std
accuracy,0.694917,0.016586
precision,0.344590,0.013983
recall,0.742114,0.019157
f1,0.470388,0.011817
roc_auc,0.784563,0.019834
pr_auc,0.457966,0.046039
balanced_accuracy,0.713255,0.008916
specificity,0.684396,0.022818


### 3.4 Save the fold models and results


In [33]:
brf_met.to_csv(OUTPUT / "brf_fold_metrics.csv", index=False)
brf_sum.to_csv(OUTPUT / "brf_summary.csv")
joblib.dump({"variant": "brf", "positive_class": 1,"requires_mice_imputed_input": True, "fold_models": brf_models}, MODEL_DIR / "brf.joblib")
print("Saved C. Balanced Random Forest fold models and results")


Saved C. Balanced Random Forest fold models and results


## Compare models

All the evaluation results are taken mean values for get the overall
metrics five folds.

Compare accuracy, precision, recall, F1-score, specificity, AUROC and PR-AUC across the three models. The comparison table displays the mean validation scores as percentages.


In [34]:
# Compare the mean validation scores of the three models.
comp = pd.DataFrame({"variant": ['Random Forest', 'Random Forest + SMOTE', 'Balanced Random Forest'],
    "accuracy": [rf_acc, rf_smote_acc, brf_acc], "precision": [rf_prec, rf_smote_prec, brf_prec], "recall": [rf_rec, rf_smote_rec, brf_rec],
    "f1": [rf_f1, rf_smote_f1, brf_f1],"roc_auc": [rf_roc_auc, rf_smote_roc_auc, brf_roc_auc], "pr_auc": [rf_pr_auc, rf_smote_pr_auc, brf_pr_auc],"balanced_accuracy": [rf_bal_acc, rf_smote_bal_acc, brf_bal_acc],
    "specificity": [rf_spec, rf_smote_spec, brf_spec],}).set_index("variant")

In [35]:
comp.to_csv(OUTPUT / "comparison_mean.csv")
display((comp * 100).style.format("{:.2f}%"))

,accuracy,precision,recall,f1,roc_auc,pr_auc,balanced_accuracy,specificity
variant,,,,,,,,
Random Forest,82.34%,69.78%,5.84%,10.63%,78.38%,46.04%,52.62%,99.40%
Random Forest + SMOTE,72.27%,36.49%,70.08%,47.99%,77.88%,44.77%,71.41%,72.75%
Balanced Random Forest,69.49%,34.46%,74.21%,47.04%,78.46%,45.80%,71.33%,68.44%


## Model Evaluation

According to the above table results, Random Forest without balancing perform the highest
accuracy of 82.34% and precision of 69.78%. But its recall for classifying the diabetes class is
only 5.84% while 99.40 % of success rate for predicting people who has no diabetes. This can be
because the original dataset 81% of Tennessee respondents belonged to the no-diabetes class.
After Appling SMOTE the baseline model RF model achieved 72.27 % accuracy and having
good recall score of around 70.08%. The Balanced random forest had lowest accuracy among
other 2 models but has the highest recall score of 74.21%. It tells us that Balanced Random
Forest is a useful alternative for diabetes classification because it achieved the highest recall
score. Although it had lower precision and f1 score compare Random forest with SMOTE. If the
main priority is identifying more diabetes cases, Balanced Random Forest would be the better
choice because it achieved the highest recall. But we need to choose a model that has a better
balance results all metrics so in that case choosing the Random Forest model used in my baseline
implementation would still be better.
